# PDF Audiobook — optional Kokoro GPU run

This notebook keeps the desktop app unchanged. It clones the project, runs the existing headless pipeline with one Kokoro narrator on the selected Colab GPU, and downloads the verified M4B. Free Colab runtimes can disconnect or have no GPU; checkpoints live under `/content` unless you change the workspace path.

**Previewing a voice.** Pick a voice and speed in the **Settings** cell, run it, then run the **Preview the selected voice** cell. It renders one short sentence and plays it inline. No PDF and no conversion are needed, so you can audition voices before committing to a long run.

**Changing settings on a run that already started.** A conversion is bound to the voice, speed, PDF and chapter settings it began with. Leave `start_new` unticked to *resume* that conversion after a disconnect — rerun the generate cell with the same settings and it picks up from the last completed chapter. To change the voice, speed, PDF or chapter settings you must tick `start_new`, which discards that conversion's checkpoints (its entire active conversion tree) and begins again from the first chapter.

In [ ]:
import subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Choose Runtime > Change runtime type > T4 GPU, then rerun.')
print(torch.cuda.get_device_name(0))

In [ ]:
# Linux tools used by the existing PDF parser and M4B finalizer.
!apt-get -qq update
!apt-get -qq install -y ffmpeg espeak-ng
!pip -q install uv

In [ ]:
# Colab's runtime Python may be 3.12; provision the project's pinned Python 3.11 environment.
REPO = '/content/AudiobookFree'
# BRANCH must name the branch that holds the runner this notebook drives.
# Change it to 'main' once this work is merged.
BRANCH = 'morefeatures'
!rm -rf /content/AudiobookFree
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/AugustZhang1/AudiobookFree.git', REPO], check=True)
!uv python install 3.11
!uv venv --python 3.11 /content/audiobook-venv
!uv pip install --python /content/audiobook-venv/bin/python -e {REPO}
!uv pip install --python /content/audiobook-venv/bin/python 'kokoro==0.9.4' 'spacy<4' 'en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0.tar.gz'
PY = '/content/audiobook-venv/bin/python'

## Settings

Choose the voice, speed and chapter handling here. The form fields only change the notebook's variables when the cell runs, so **rerun this cell after changing any value** — otherwise the preview and generate cells below still use the previous values.

If a conversion has already started, changing any of these values also requires ticking `start_new` in this cell, which discards that conversion's checkpoints. Leave `start_new` unticked to resume an interrupted run with the settings it started with.

In [ ]:
voice = 'af_heart' # @param ["af_heart", "af_alloy", "af_aoede", "af_bella", "af_jessica", "af_kore", "af_nicole", "af_nova", "af_river", "af_sarah", "af_sky", "am_adam", "am_echo", "am_eric", "am_fenrir", "am_liam", "am_michael", "am_onyx", "am_puck", "am_santa", "bf_alice", "bf_emma", "bf_isabella", "bf_lily", "bm_daniel", "bm_fable", "bm_george", "bm_lewis"]
speed = 1.0 # @param {type:"number"}
chapter_mode = 'original' # @param ["original", "whole", "custom"]
chapter_count = 4 # @param {type:"integer"}
start_new = False # @param {type:"boolean"}
workspace = '/content/pdf-audiobook-workspace'
output_dir = '/content/pdf-audiobook-output'
print(f'voice={voice} speed={speed} chapter_mode={chapter_mode} chapter_count={chapter_count} start_new={start_new} workspace={workspace} output_dir={output_dir}')

## Preview the selected voice (optional)

Renders one short sentence with the voice and speed chosen above and plays it inline. It needs no PDF and starts no conversion, so it never touches your workspace or checkpoints.

To compare voices, edit the **Settings** cell, rerun it, then rerun this cell — the preview always uses the values from the last run of the settings cell.

The first preview also downloads the Kokoro model, so it takes noticeably longer than later ones.

In [ ]:
from IPython.display import Audio, display
preview_path = f'/content/voice-previews/{voice}-{speed}.wav'
preview = subprocess.run(
    [PY, '-u', '-m', 'pdf_audiobook.colab', '--preview-out', preview_path, '--voice', voice, '--speed', str(speed)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(preview.stdout, end='')
if preview.returncode != 0:
    raise RuntimeError('Voice preview failed; see the output above.')
display(Audio(filename=preview_path))

In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload one selectable-text PDF.')
pdf_name = next(iter(uploaded))
pdf_path = '/content/' + pdf_name

In [ ]:
# One long, resumable headless run. Rerun this cell after a disconnect with the same PDF/settings.
count = chapter_count if chapter_mode == 'custom' else None
cmd = [PY, '-u', '-m', 'pdf_audiobook.colab', pdf_path, '--workspace-root', workspace, '--output-dir', output_dir, '--voice', voice, '--speed', str(speed), '--chapter-mode', chapter_mode]
if count is not None:
    cmd += ['--chapter-count', str(count)]
if start_new:
    cmd += ['--start-new']
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
verified_path = None
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    if line.startswith('Verified M4B: '):
        verified_path = line.removeprefix('Verified M4B: ').strip()
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Colab conversion failed with exit code {return_code}')
if not verified_path:
    raise RuntimeError('The runner did not report a verified M4B path.')
files.download(verified_path)